# Student Feedback Insight & Action Recommendation System

### NLP Mini Project

**Project idea:** Instead of only predicting Positive/Neutral/Negative sentiment, this system converts student feedback into **actionable insights** by combining sentiment analysis, feedback-topic identification, priority detection, and improvement recommendations.

**Dataset:** Student feedback CSV with `feedback` and `sentiment` columns.

**Main NLP/ML techniques:** Text preprocessing, TF-IDF, Linear SVM (LinearSVC), sentiment classification, keyword-based topic identification, priority scoring, and recommendation generation.

**Name:** Zaid Seliya
**UIN:** 231A050

## 1. Problem Statement

Educational institutions receive many student comments about teachers, courses, practical sessions, and other academic experiences. Reading every response manually makes it difficult to quickly identify the most important issues.

The proposed system analyzes each feedback comment and produces:
- Sentiment: Positive, Neutral, or Negative
- Topic: Teacher, Course, Practical, Infrastructure, or General
- Priority: High, Medium, or Low
- Suggested action/recommendation

This makes the project more useful than a basic three-class sentiment classifier because the output is designed to support decision-making.

## 2. Objectives

1. Preprocess student feedback text.
2. Train an NLP sentiment classifier using TF-IDF and Linear SVM.
3. Identify the main topic of each feedback comment.
4. Assign a priority level to feedback.
5. Generate a simple action recommendation.
6. Create summary statistics and visualizations for administrators.
7. Allow new feedback to be analyzed through a reusable prediction function.

## 3. Solution Approach

**CSV → Text Cleaning → TF-IDF Features → Linear SVM → Sentiment**

In parallel:

**Feedback Text → Topic Identification → Priority Score → Recommendation**

Finally, the system combines these outputs into an **actionable feedback report**.

In [ ]:
# STEP 1: Import libraries

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")


In [ ]:
# STEP 2: Upload and load the CSV dataset

from google.colab import files

uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload a CSV file.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print("Loaded:", file_name)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# STEP 3: Check dataset structure

print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

if not {"feedback", "sentiment"}.issubset(df.columns):
    raise ValueError("The CSV must contain 'feedback' and 'sentiment' columns.")

df = df.dropna(subset=["feedback", "sentiment"]).copy()
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.title()

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())


In [ ]:
# STEP 4: Text preprocessing

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)       # remove punctuation/numbers
    text = re.sub(r"\s+", " ", text).strip()       # remove extra spaces
    return text

df["clean_feedback"] = df["feedback"].apply(clean_text)

display(df[["feedback", "clean_feedback"]].head(10))


In [ ]:
# STEP 5: Train/test split

X = df["clean_feedback"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
# STEP 6: TF-IDF feature extraction

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training matrix:", X_train_tfidf.shape)


In [ ]:
# STEP 7: Train Linear SVM sentiment model

model = LinearSVC()
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print(f"Sentiment Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# STEP 8: Confusion matrix

labels = ["Positive", "Neutral", "Negative"]
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Sentiment Classification Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()


## 4. Feedback Topic Identification

The dataset contains only feedback text and sentiment labels, so topic identification is implemented as a transparent **keyword-based NLP layer**.

Topics used:
- Teacher
- Course
- Practical
- Infrastructure
- General

This is an additional analysis layer and is not claimed as a separately trained topic-classification model.

In [ ]:
# STEP 9: Topic identification

topic_keywords = {
    "Teacher": [
        "teacher", "faculty", "professor", "teaching", "explain",
        "explanation", "lecture", "lectures"
    ],
    "Course": [
        "course", "subject", "syllabus", "unit", "topic",
        "topics", "semester"
    ],
    "Practical": [
        "practical", "lab", "laboratory", "experiment",
        "programming", "coding"
    ],
    "Infrastructure": [
        "classroom", "infrastructure", "computer", "computers",
        "equipment", "facility", "facilities"
    ]
}

def detect_topic(text):
    text = clean_text(text)
    scores = {}

    for topic, words in topic_keywords.items():
        scores[topic] = sum(1 for word in words if word in text)

    best_topic = max(scores, key=scores.get)
    return best_topic if scores[best_topic] > 0 else "General"

df["topic"] = df["feedback"].apply(detect_topic)

print(df["topic"].value_counts())
display(df[["feedback", "sentiment", "topic"]].head(10))


In [ ]:
# STEP 10: Priority detection

high_priority_words = [
    "poor", "bad", "worst", "confusing", "unclear", "problem",
    "issue", "difficult", "not clear", "very slow", "lack"
]

medium_priority_words = [
    "average", "okay", "improve", "improvement", "moderate"
]

def detect_priority(text, sentiment):
    t = clean_text(text)

    if sentiment == "Negative" or any(word in t for word in high_priority_words):
        return "High"
    if sentiment == "Neutral" or any(word in t for word in medium_priority_words):
        return "Medium"
    return "Low"

df["priority"] = [
    detect_priority(text, sentiment)
    for text, sentiment in zip(df["feedback"], df["sentiment"])
]

print(df["priority"].value_counts())


In [ ]:
# STEP 11: Generate action recommendations

recommendations = {
    ("Teacher", "Negative"): "Review teaching clarity and provide additional explanations.",
    ("Teacher", "Neutral"): "Collect specific suggestions and improve explanation where needed.",
    ("Teacher", "Positive"): "Maintain the effective teaching practices.",

    ("Course", "Negative"): "Review course content, difficulty, and topic coverage.",
    ("Course", "Neutral"): "Collect suggestions to improve course structure and content.",
    ("Course", "Positive"): "Continue the current course structure and content.",

    ("Practical", "Negative"): "Review practical instructions, experiments, and lab support.",
    ("Practical", "Neutral"): "Improve practical guidance and collect detailed lab feedback.",
    ("Practical", "Positive"): "Continue the effective practical activities.",

    ("Infrastructure", "Negative"): "Inspect facilities and address the reported infrastructure issue.",
    ("Infrastructure", "Neutral"): "Review facilities and identify possible improvements.",
    ("Infrastructure", "Positive"): "Maintain the current facilities and resources.",

    ("General", "Negative"): "Review the feedback and investigate the underlying concern.",
    ("General", "Neutral"): "Collect more specific feedback for improvement.",
    ("General", "Positive"): "Maintain the positive aspects mentioned by students."
}

def make_recommendation(topic, sentiment):
    return recommendations.get(
        (topic, sentiment),
        "Review the feedback and take an appropriate action."
    )

df["recommendation"] = [
    make_recommendation(topic, sentiment)
    for topic, sentiment in zip(df["topic"], df["sentiment"])
]

display(df[["feedback", "sentiment", "topic", "priority", "recommendation"]].head(15))


In [ ]:
# STEP 12: Create the final insight report

insight_report = df[
    ["feedback", "sentiment", "topic", "priority", "recommendation"]
].copy()

print("Final insight report created.")
display(insight_report.head(20))

# Save the generated report
insight_report.to_csv("student_feedback_insight_report.csv", index=False)
print("Saved as: student_feedback_insight_report.csv")


In [ ]:
# STEP 13: Visualize the major insights

# Sentiment distribution
sentiment_counts = df["sentiment"].value_counts()

plt.figure(figsize=(7, 4))
sentiment_counts.plot(kind="bar")
plt.title("Student Feedback by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of Feedbacks")
plt.xticks(rotation=0)
plt.show()

# Topic distribution
topic_counts = df["topic"].value_counts()

plt.figure(figsize=(7, 4))
topic_counts.plot(kind="bar")
plt.title("Student Feedback by Topic")
plt.xlabel("Topic")
plt.ylabel("Number of Feedbacks")
plt.xticks(rotation=30)
plt.show()

# Priority distribution
priority_counts = df["priority"].value_counts()

plt.figure(figsize=(7, 4))
priority_counts.plot(kind="bar")
plt.title("Feedback Priority Distribution")
plt.xlabel("Priority")
plt.ylabel("Number of Feedbacks")
plt.xticks(rotation=0)
plt.show()


In [ ]:
# STEP 14: Topic vs sentiment analysis

topic_sentiment = pd.crosstab(df["topic"], df["sentiment"])
print("Topic vs Sentiment:")
display(topic_sentiment)

plt.figure(figsize=(8, 5))
topic_sentiment.plot(kind="bar", ax=plt.gca())
plt.title("Topic-wise Sentiment Distribution")
plt.xlabel("Topic")
plt.ylabel("Number of Feedbacks")
plt.xticks(rotation=30)
plt.legend(title="Sentiment")
plt.show()


## 5. Analyze New Student Feedback

The following function combines the trained sentiment model with topic, priority, and recommendation analysis.

**Example output:** a new comment is converted into an actionable result rather than only a sentiment label.

In [ ]:
# STEP 15: Complete analyzer for new feedback

def analyze_feedback(feedback):
    cleaned = clean_text(feedback)
    features = vectorizer.transform([cleaned])

    sentiment = model.predict(features)[0]
    topic = detect_topic(feedback)
    priority = detect_priority(feedback, sentiment)
    recommendation = make_recommendation(topic, sentiment)

    return {
        "Feedback": feedback,
        "Sentiment": sentiment,
        "Topic": topic,
        "Priority": priority,
        "Recommendation": recommendation
    }

# Test examples
examples = [
    "The teacher explains every topic clearly and the lectures are excellent.",
    "The course is okay but some topics need improvement.",
    "The practical sessions are confusing and the lab instructions are unclear."
]

for text in examples:
    print(analyze_feedback(text))
    print("-" * 80)


In [ ]:
# STEP 16: Enter your own feedback

user_feedback = input("Enter student feedback: ")

if user_feedback.strip():
    result = analyze_feedback(user_feedback)
    print("\n--- Feedback Analysis ---")
    for key, value in result.items():
        print(f"{key}: {value}")
else:
    print("No feedback entered.")


## 6. Conclusion

The **Student Feedback Insight & Action Recommendation System** extends basic sentiment analysis into an actionable NLP solution. It uses TF-IDF and Linear SVM to classify sentiment and adds topic identification, priority detection, and recommendation generation.

The final report helps an institution understand **what students are saying, what area it concerns, how important it is, and what action can be considered**.

### Future Scope
- Use a larger real-world college feedback dataset.
- Add multilingual/Hinglish support.
- Replace keyword topic detection with a trained topic-classification model.
- Compare multiple ML and deep-learning models.
- Build a Streamlit dashboard for real-time feedback analysis.
- Add department-wise and subject-wise reporting when those fields are available in the dataset.
